# Fix OCR Dataset
On 2026-07-21, it was noticed that the OCR-generated dataset could benefit from containing information on which split (e.g., train, test, val) the card belongs to. This ensures consistent dataset training with the non-OCR version of the model. This notebook was designed to modify that file so that it contains the split information.

In [1]:
## packages

### file manipulation
import json

### link directory
from pathlib import Path
import sys

workspace_root = Path.cwd()
if workspace_root.name == 'notebooks':
    workspace_root = workspace_root.parent
if str(workspace_root) not in sys.path:
    sys.path.append(str(workspace_root))

### custom packages
from src.ocr.input_dataset import load_manifest_records, summarize_manifest

In [3]:
from src.config import OCR_FILENAME

In [2]:
## data
manifest_path = workspace_root / "data" / "card_image_text_manifest.jsonl"
records = load_manifest_records(manifest_path, skip_missing_images=True)
summary = summarize_manifest(records)
print(f'Input Data Summary:')
for k, v in summary.items():
    print(f'\t{k}: {v}')

Input Data Summary:
	num_records: 8443
	split_counts: {'train': 6713, 'val': 1680, 'test': 50}


In [12]:
## read in the OCR generated text
ocr_path = workspace_root / 'data' / OCR_FILENAME
with open(ocr_path, 'r', encoding = 'utf-8') as f:
    ocr_data = json.load(f)

print(len(ocr_data))

8443


## Add Split to OCR Data

In [14]:
records[0]

{'id': 8079,
 'oracle_id': '4317f4f1-d339-4940-b46e-659641035595',
 'card_name': 'Ascended Lawmage',
 'split': 'train',
 'image_path': '/Users/nickcruickshank/Projects/mtg-multimodal-classification/data/card_images/4317f4f1-d339-4940-b46e-659641035595.png',
 'image_type': 'png',
 'image_exists': True,
 'target_text': "Ascended Lawmage\n        Mana Cost = {2}{W}{U}\nMana Value = 4.0\n\n        Type Line = Creature — Vedalken Wizard\n\n        Rules Text = Flying\nHexproof (This creature can't be the target of spells or abilities your opponents control.)\n\n        Power = 3\nToughness = 2\n\n\n        Color Identity = ['U', 'W']\n\n        Rarity = uncommon",
 'tags': ['evasion', 'french vanilla']}

In [13]:
ocr_data[0]

{'id': 8079,
 'oracle_id': '4317f4f1-d339-4940-b46e-659641035595',
 'card_name': 'Ascended Lawmage',
 'card_text_ocr': "Ascended Lawmage\n\nCreature - Vedalken Wizard\n\nFlying\n\nHexproof f (This creature can't be the target of spells or abilities your opponents control.)\n\nA lawmage's runic script is an act c of governance given form: legislation written directly onto the air itself.\n\n3/2",
 'tags': ['evasion', 'french vanilla']}

In [20]:
output = []
for card in ocr_data:
    # get the record corresponding to the card
    rec = [r for r in records if r['oracle_id'] == card['oracle_id']][0]

    # update the card
    card_update = card.copy()
    card_update['split'] = rec['split']

    # store out
    output.append(card_update)

## Save New Output

In [21]:
out_path = workspace_root / "data" / OCR_FILENAME
with open(out_path, 'w', encoding = 'utf-8') as f:
    json.dump(output, f, indent = 4)
print(f'Docling generated card text saved to {str(out_path)}')

Docling generated card text saved to /Users/nickcruickshank/Projects/mtg-multimodal-classification/data/card_image_ocr_text_tags.json
